# Wolf Sheep Simple 3 — Modelo Ampliado

Este notebook amplia o modelo **Wolf Sheep Simple 3** com recursos adicionais inspirados em Sistemas Complexos, Ecologia Computacional e modelos baseados em agentes.

## Recursos adicionados

- Idade dos agentes.
- Fome máxima.
- Visão dos lobos.
- Estratégia de fuga das ovelhas.
- Reprodução dependente de energia.
- Obstáculos no ambiente.
- Estações do ano.
- Doenças.
- Migração.
- Comparação com Lotka-Volterra.
- Exportação de imagens por tick.
- Protótipo de dashboard com Streamlit.

A simulação foi escrita em Python puro, sem depender de NetLogo, `nl4py` ou fontes de emoji.

## 1. Introdução a Sistemas Complexos

Sistemas complexos são formados por muitos elementos que interagem localmente. Mesmo quando cada agente segue regras simples, o comportamento coletivo pode ser não linear, emergente e difícil de prever.

Neste modelo:

- A **grama** é o recurso primário.
- As **ovelhas** consomem grama, fogem de lobos, envelhecem, adoecem, migram e se reproduzem.
- Os **lobos** procuram ovelhas dentro de um raio de visão, caçam, envelhecem, adoecem, migram e se reproduzem.
- O ambiente possui **obstáculos**, que bloqueiam movimento e crescimento de grama.
- As **estações do ano** alteram a taxa de crescimento da grama.
- A dinâmica é comparada com o modelo matemático clássico **Lotka-Volterra**.

In [ ]:
import os
import math
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

## 2. Parâmetros do modelo

Os parâmetros abaixo controlam o comportamento geral da simulação.
Eles funcionam de maneira semelhante aos controles do NetLogo.

In [ ]:
params_base = {
    # Ambiente
    "grid_size": 50,
    "grass_max": 10.0,
    "grass_regrowth_base": 0.45,
    "obstacle_density": 0.06,

    # Populações iniciais
    "initial_sheep": 160,
    "initial_wolves": 45,

    # Energia inicial
    "initial_sheep_energy": 24,
    "initial_wolf_energy": 36,

    # Energia e fome
    "sheep_movement_cost": 1.0,
    "wolf_movement_cost": 1.2,
    "energy_gain_from_grass": 4.0,
    "energy_gain_from_sheep": 22.0,
    "max_hunger_sheep": 18,
    "max_hunger_wolf": 24,

    # Reprodução dependente de energia
    "sheep_reproduction_energy_threshold": 34,
    "wolf_reproduction_energy_threshold": 48,
    "sheep_reproduction_prob": 0.06,
    "wolf_reproduction_prob": 0.035,
    "offspring_energy_fraction": 0.45,

    # Idade
    "max_age_sheep": 90,
    "max_age_wolf": 120,

    # Visão e comportamento
    "wolf_vision": 5,
    "sheep_vision": 4,

    # Doença
    "initial_disease_prob": 0.02,
    "disease_spread_prob": 0.06,
    "disease_energy_cost": 1.2,
    "disease_death_prob": 0.01,
    "disease_recovery_prob": 0.015,

    # Migração
    "migration_prob": 0.01,
    "migration_edge_bias": 0.75,

    # Estações
    "season_length": 50,
    "season_growth_multipliers": {
        "primavera": 1.25,
        "verao": 1.00,
        "outono": 0.70,
        "inverno": 0.35
    },

    # Simulação
    "n_steps": 300,
    "random_seed": 42,
}

## 3. Coordenadas X e Y

O ambiente é uma grade bidimensional formada por células chamadas aqui de **patches**.

Cada agente possui:

- `x`: coordenada horizontal, isto é, sua posição no eixo leste-oeste.
- `y`: coordenada vertical, isto é, sua posição no eixo norte-sul.

A posição `(x, y)` identifica exatamente o patch onde o agente está localizado.

Exemplo: `(10, 25)` significa que o agente está na coluna 10 e na linha 25 do ambiente.

O mundo é **toroidal**: se um agente ultrapassa uma borda, ele reaparece do lado oposto, como ocorre em muitos modelos do NetLogo.

## 4. Classe dos agentes

Agora os agentes possuem atributos adicionais:

- `age`: idade.
- `hunger`: tempo acumulado sem alimentação.
- `sick`: indicador de doença.
- `kind`: tipo do agente.
- `energy`: energia disponível.

In [ ]:
@dataclass
class Agent:
    x: int
    y: int
    energy: float
    kind: str
    age: int = 0
    hunger: int = 0
    sick: bool = False

    def position(self):
        return (self.x, self.y)

    def copy(self):
        return Agent(
            x=self.x,
            y=self.y,
            energy=self.energy,
            kind=self.kind,
            age=self.age,
            hunger=self.hunger,
            sick=self.sick
        )

## 5. Funções auxiliares espaciais

Estas funções calculam distância, vizinhança, obstáculos e movimentos possíveis.

In [ ]:
def toroidal_distance(a, b, size):
    dx = abs(a[0] - b[0])
    dy = abs(a[1] - b[1])
    dx = min(dx, size - dx)
    dy = min(dy, size - dy)
    return math.sqrt(dx * dx + dy * dy)


def wrap_position(x, y, size):
    return x % size, y % size


def is_free(x, y, obstacles):
    return not obstacles[x, y]


def random_neighbor(x, y, size, obstacles):
    candidates = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0:
                continue
            nx, ny = wrap_position(x + dx, y + dy, size)
            if is_free(nx, ny, obstacles):
                candidates.append((nx, ny))

    if candidates:
        return random.choice(candidates)

    return x, y


def step_towards(agent, target_pos, size, obstacles):
    best_pos = (agent.x, agent.y)
    best_dist = toroidal_distance(best_pos, target_pos, size)

    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            nx, ny = wrap_position(agent.x + dx, agent.y + dy, size)
            if not is_free(nx, ny, obstacles):
                continue

            dist = toroidal_distance((nx, ny), target_pos, size)
            if dist < best_dist:
                best_dist = dist
                best_pos = (nx, ny)

    return best_pos


def step_away(agent, threat_pos, size, obstacles):
    best_pos = (agent.x, agent.y)
    best_dist = toroidal_distance(best_pos, threat_pos, size)

    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            nx, ny = wrap_position(agent.x + dx, agent.y + dy, size)
            if not is_free(nx, ny, obstacles):
                continue

            dist = toroidal_distance((nx, ny), threat_pos, size)
            if dist > best_dist:
                best_dist = dist
                best_pos = (nx, ny)

    return best_pos


def nearest_agent(agent, candidates, vision, size):
    nearest = None
    nearest_dist = float("inf")

    for other in candidates:
        dist = toroidal_distance(agent.position(), other.position(), size)
        if dist <= vision and dist < nearest_dist:
            nearest = other
            nearest_dist = dist

    return nearest, nearest_dist

## 6. Estações do ano

As estações alteram a taxa de crescimento da grama:

- Primavera: crescimento maior.
- Verão: crescimento normal.
- Outono: crescimento reduzido.
- Inverno: crescimento baixo.

In [ ]:
def get_season(tick, params):
    seasons = ["primavera", "verao", "outono", "inverno"]
    season_length = params["season_length"]
    idx = (tick // season_length) % len(seasons)
    return seasons[idx]


def current_grass_growth_rate(tick, params):
    season = get_season(tick, params)
    multiplier = params["season_growth_multipliers"][season]
    return params["grass_regrowth_base"] * multiplier

## 7. Inicialização do modelo

A função `setup()` cria:

- A matriz de grama.
- A matriz de obstáculos.
- As ovelhas.
- Os lobos.
- Doença inicial em uma pequena fração da população.

In [ ]:
def random_free_position(size, obstacles):
    for _ in range(size * size * 2):
        x = random.randrange(size)
        y = random.randrange(size)
        if not obstacles[x, y]:
            return x, y
    return 0, 0


def setup(params):
    np.random.seed(params["random_seed"])
    random.seed(params["random_seed"])

    size = params["grid_size"]

    obstacles = np.random.random((size, size)) < params["obstacle_density"]

    grass = np.random.uniform(
        low=0,
        high=params["grass_max"],
        size=(size, size)
    )

    grass[obstacles] = 0

    sheep = []
    for _ in range(params["initial_sheep"]):
        x, y = random_free_position(size, obstacles)
        sheep.append(
            Agent(
                x=x,
                y=y,
                energy=params["initial_sheep_energy"],
                kind="sheep",
                sick=random.random() < params["initial_disease_prob"]
            )
        )

    wolves = []
    for _ in range(params["initial_wolves"]):
        x, y = random_free_position(size, obstacles)
        wolves.append(
            Agent(
                x=x,
                y=y,
                energy=params["initial_wolf_energy"],
                kind="wolf",
                sick=random.random() < params["initial_disease_prob"]
            )
        )

    return grass, obstacles, sheep, wolves

## 8. Visualização do mundo

A visualização não usa emojis para evitar o erro de fonte `Glyph missing from font(s) DejaVu Sans`.

Representação:

- Fundo verde: grama.
- Células pretas: obstáculos.
- Ovelhas saudáveis: círculos brancos.
- Ovelhas doentes: círculos amarelos.
- Lobos saudáveis: estrelas vermelhas.
- Lobos doentes: estrelas roxas.

In [ ]:
def plot_world(
    grass,
    obstacles,
    sheep,
    wolves,
    tick=0,
    params=params_base,
    title="Wolf Sheep Simple 3 ampliado",
    save_path=None
):
    fig, ax = plt.subplots(figsize=(9, 9))

    grass_img = ax.imshow(
        grass.T,
        origin="lower",
        cmap="Greens",
        vmin=0,
        vmax=params["grass_max"],
        alpha=0.95
    )

    ox, oy = np.where(obstacles)
    if len(ox) > 0:
        ax.scatter(
            ox, oy,
            s=22,
            c="black",
            marker="s",
            label="Obstáculos",
            zorder=2
        )

    healthy_sheep = [s for s in sheep if not s.sick]
    sick_sheep = [s for s in sheep if s.sick]
    healthy_wolves = [w for w in wolves if not w.sick]
    sick_wolves = [w for w in wolves if w.sick]

    if healthy_sheep:
        ax.scatter(
            [s.x for s in healthy_sheep],
            [s.y for s in healthy_sheep],
            s=35,
            c="white",
            edgecolors="black",
            linewidths=0.6,
            marker="o",
            label=f"Ovelhas saudáveis ({len(healthy_sheep)})",
            zorder=3
        )

    if sick_sheep:
        ax.scatter(
            [s.x for s in sick_sheep],
            [s.y for s in sick_sheep],
            s=40,
            c="yellow",
            edgecolors="black",
            linewidths=0.7,
            marker="o",
            label=f"Ovelhas doentes ({len(sick_sheep)})",
            zorder=4
        )

    if healthy_wolves:
        ax.scatter(
            [w.x for w in healthy_wolves],
            [w.y for w in healthy_wolves],
            s=90,
            c="red",
            edgecolors="black",
            linewidths=0.8,
            marker="*",
            label=f"Lobos saudáveis ({len(healthy_wolves)})",
            zorder=5
        )

    if sick_wolves:
        ax.scatter(
            [w.x for w in sick_wolves],
            [w.y for w in sick_wolves],
            s=100,
            c="purple",
            edgecolors="black",
            linewidths=0.8,
            marker="*",
            label=f"Lobos doentes ({len(sick_wolves)})",
            zorder=6
        )

    season = get_season(tick, params)
    ax.set_title(f"{title} — tick {tick} — estação: {season}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Coordenada X — posição horizontal no ambiente")
    ax.set_ylabel("Coordenada Y — posição vertical no ambiente")
    ax.set_xlim(-0.5, grass.shape[0] - 0.5)
    ax.set_ylim(-0.5, grass.shape[1] - 0.5)
    ax.legend(loc="upper right", framealpha=0.93, fontsize=8)

    cbar = plt.colorbar(grass_img, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Quantidade de grama")

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=140, bbox_inches="tight")

    plt.show()

## 9. Migração

A migração permite entrada de novos agentes pelas bordas do mundo.

Ela representa:

- Chegada de indivíduos de ambientes vizinhos.
- Dispersão espacial.
- Colonização de áreas livres.

In [ ]:
def maybe_migrate_agents(sheep, wolves, obstacles, params):
    size = params["grid_size"]

    def edge_position():
        if random.random() < params["migration_edge_bias"]:
            side = random.choice(["left", "right", "bottom", "top"])
            if side == "left":
                return 0, random.randrange(size)
            if side == "right":
                return size - 1, random.randrange(size)
            if side == "bottom":
                return random.randrange(size), 0
            return random.randrange(size), size - 1
        return random_free_position(size, obstacles)

    if random.random() < params["migration_prob"]:
        x, y = edge_position()
        if not obstacles[x, y]:
            sheep.append(Agent(x, y, params["initial_sheep_energy"], "sheep"))

    if random.random() < params["migration_prob"] * 0.6:
        x, y = edge_position()
        if not obstacles[x, y]:
            wolves.append(Agent(x, y, params["initial_wolf_energy"], "wolf"))

    return sheep, wolves

## 10. Doenças

A doença reduz energia, pode se espalhar entre agentes próximos e pode causar morte.

Ela adiciona uma camada epidemiológica ao sistema predador-presa.

In [ ]:
def update_disease(agents, params):
    size = params["grid_size"]

    sick_positions = [a.position() for a in agents if a.sick]

    for a in agents:
        if a.sick:
            a.energy -= params["disease_energy_cost"]

            if random.random() < params["disease_death_prob"]:
                a.energy = -1

            if random.random() < params["disease_recovery_prob"]:
                a.sick = False

        else:
            for pos in sick_positions:
                if toroidal_distance(a.position(), pos, size) <= 1.5:
                    if random.random() < params["disease_spread_prob"]:
                        a.sick = True
                        break

    return agents

## 11. Regras das ovelhas

As ovelhas agora:

1. Envelhecem.
2. Fogem se detectarem lobos dentro do raio de visão.
3. Movem-se aleatoriamente se não houver ameaça.
4. Gastam energia.
5. Comem grama.
6. Acumulam fome se não comerem.
7. Reproduzem apenas se tiverem energia suficiente.
8. Podem morrer por idade, energia, fome ou doença.

In [ ]:
def update_sheep(sheep, wolves, grass, obstacles, params):
    size = params["grid_size"]
    survivors = []
    newborns = []

    for s in sheep:
        s.age += 1

        nearest_wolf, dist = nearest_agent(
            s,
            wolves,
            params["sheep_vision"],
            size
        )

        if nearest_wolf is not None:
            nx, ny = step_away(s, nearest_wolf.position(), size, obstacles)
        else:
            nx, ny = random_neighbor(s.x, s.y, size, obstacles)

        s.x, s.y = nx, ny
        s.energy -= params["sheep_movement_cost"]

        ate = False
        if grass[s.x, s.y] >= params["energy_gain_from_grass"]:
            grass[s.x, s.y] -= params["energy_gain_from_grass"]
            s.energy += params["energy_gain_from_grass"]
            s.hunger = 0
            ate = True

        if not ate:
            s.hunger += 1

        if (
            s.energy >= params["sheep_reproduction_energy_threshold"]
            and random.random() < params["sheep_reproduction_prob"]
        ):
            child_energy = s.energy * params["offspring_energy_fraction"]
            s.energy -= child_energy
            newborns.append(
                Agent(
                    x=s.x,
                    y=s.y,
                    energy=child_energy,
                    kind="sheep",
                    age=0,
                    hunger=0,
                    sick=s.sick and random.random() < 0.25
                )
            )

        alive = (
            s.energy > 0
            and s.hunger <= params["max_hunger_sheep"]
            and s.age <= params["max_age_sheep"]
        )

        if alive:
            survivors.append(s)

    survivors.extend(newborns)
    return survivors, grass

## 12. Regras dos lobos

Os lobos agora:

1. Envelhecem.
2. Procuram ovelhas dentro de um raio de visão.
3. Movem-se em direção à presa mais próxima.
4. Movem-se aleatoriamente quando não enxergam presa.
5. Gastam energia.
6. Comem ovelhas no mesmo patch.
7. Acumulam fome se não comem.
8. Reproduzem apenas se tiverem energia suficiente.
9. Podem morrer por idade, energia, fome ou doença.

In [ ]:
def update_wolves(wolves, sheep, obstacles, params):
    size = params["grid_size"]

    sheep_by_position = {}
    for idx, s in enumerate(sheep):
        sheep_by_position.setdefault(s.position(), []).append(idx)

    eaten_sheep = set()
    survivors = []
    newborns = []

    for w in wolves:
        w.age += 1

        available_sheep = [
            s for idx, s in enumerate(sheep)
            if idx not in eaten_sheep
        ]

        target, dist = nearest_agent(
            w,
            available_sheep,
            params["wolf_vision"],
            size
        )

        if target is not None:
            nx, ny = step_towards(w, target.position(), size, obstacles)
        else:
            nx, ny = random_neighbor(w.x, w.y, size, obstacles)

        w.x, w.y = nx, ny
        w.energy -= params["wolf_movement_cost"]

        ate = False
        pos = w.position()

        if pos in sheep_by_position:
            available_here = [
                idx for idx in sheep_by_position[pos]
                if idx not in eaten_sheep
            ]

            if available_here:
                chosen_idx = available_here[0]
                eaten_sheep.add(chosen_idx)
                w.energy += params["energy_gain_from_sheep"]
                w.hunger = 0
                ate = True

        if not ate:
            w.hunger += 1

        if (
            w.energy >= params["wolf_reproduction_energy_threshold"]
            and random.random() < params["wolf_reproduction_prob"]
        ):
            child_energy = w.energy * params["offspring_energy_fraction"]
            w.energy -= child_energy
            newborns.append(
                Agent(
                    x=w.x,
                    y=w.y,
                    energy=child_energy,
                    kind="wolf",
                    age=0,
                    hunger=0,
                    sick=w.sick and random.random() < 0.25
                )
            )

        alive = (
            w.energy > 0
            and w.hunger <= params["max_hunger_wolf"]
            and w.age <= params["max_age_wolf"]
        )

        if alive:
            survivors.append(w)

    remaining_sheep = [
        s for idx, s in enumerate(sheep)
        if idx not in eaten_sheep
    ]

    survivors.extend(newborns)
    return survivors, remaining_sheep, len(eaten_sheep)

## 13. Crescimento da grama com estações e obstáculos

A grama cresce conforme a estação do ano e não cresce sobre obstáculos.

In [ ]:
def regrow_grass(grass, obstacles, tick, params):
    rate = current_grass_growth_rate(tick, params)
    grass = grass + rate
    grass[grass > params["grass_max"]] = params["grass_max"]
    grass[obstacles] = 0
    return grass

## 14. Métricas ecológicas

Registramos métricas populacionais, energéticas, epidemiológicas e ambientais.

In [ ]:
def collect_metrics(tick, grass, obstacles, sheep, wolves, eaten_sheep, params):
    sheep_energy = [s.energy for s in sheep]
    wolf_energy = [w.energy for w in wolves]

    sick_sheep = sum(1 for s in sheep if s.sick)
    sick_wolves = sum(1 for w in wolves if w.sick)

    free_cells = np.logical_not(obstacles)
    mean_grass_free = float(np.mean(grass[free_cells])) if np.any(free_cells) else 0.0

    return {
        "tick": tick,
        "season": get_season(tick, params),
        "sheep": len(sheep),
        "wolves": len(wolves),
        "sick_sheep": sick_sheep,
        "sick_wolves": sick_wolves,
        "disease_prevalence_sheep": sick_sheep / len(sheep) if sheep else 0,
        "disease_prevalence_wolves": sick_wolves / len(wolves) if wolves else 0,
        "mean_grass": float(np.mean(grass)),
        "mean_grass_free_cells": mean_grass_free,
        "grass_growth_rate": current_grass_growth_rate(tick, params),
        "mean_sheep_energy": float(np.mean(sheep_energy)) if sheep_energy else 0,
        "mean_wolf_energy": float(np.mean(wolf_energy)) if wolf_energy else 0,
        "mean_sheep_age": float(np.mean([s.age for s in sheep])) if sheep else 0,
        "mean_wolf_age": float(np.mean([w.age for w in wolves])) if wolves else 0,
        "mean_sheep_hunger": float(np.mean([s.hunger for s in sheep])) if sheep else 0,
        "mean_wolf_hunger": float(np.mean([w.hunger for w in wolves])) if wolves else 0,
        "eaten_sheep": eaten_sheep,
        "obstacle_ratio": float(np.mean(obstacles)),
        "sheep_extinct": len(sheep) == 0,
        "wolves_extinct": len(wolves) == 0,
    }

## 15. Execução da simulação

A função abaixo executa a simulação completa e pode também exportar imagens por tick.

In [ ]:
def copy_agents(agents):
    return [a.copy() for a in agents]


def run_simulation(
    params,
    save_frames=True,
    frame_interval=5,
    export_images=False,
    image_dir="wolf_sheep_frames"
):
    grass, obstacles, sheep, wolves = setup(params)

    if export_images:
        Path(image_dir).mkdir(parents=True, exist_ok=True)

    history = []
    frames = []

    for tick in range(params["n_steps"] + 1):
        eaten = 0

        history.append(
            collect_metrics(
                tick=tick,
                grass=grass,
                obstacles=obstacles,
                sheep=sheep,
                wolves=wolves,
                eaten_sheep=eaten,
                params=params
            )
        )

        if save_frames and tick % frame_interval == 0:
            frames.append({
                "tick": tick,
                "grass": grass.copy(),
                "obstacles": obstacles.copy(),
                "sheep": copy_agents(sheep),
                "wolves": copy_agents(wolves)
            })

        if export_images and tick % frame_interval == 0:
            path = Path(image_dir) / f"tick_{tick:04d}.png"
            plot_world(
                grass=grass,
                obstacles=obstacles,
                sheep=sheep,
                wolves=wolves,
                tick=tick,
                params=params,
                save_path=path
            )
            plt.close("all")

        if len(sheep) == 0 and len(wolves) == 0:
            break

        sheep, grass = update_sheep(sheep, wolves, grass, obstacles, params)
        wolves, sheep, eaten = update_wolves(wolves, sheep, obstacles, params)

        sheep = update_disease(sheep, params)
        wolves = update_disease(wolves, params)

        sheep = [s for s in sheep if s.energy > 0]
        wolves = [w for w in wolves if w.energy > 0]

        sheep, wolves = maybe_migrate_agents(sheep, wolves, obstacles, params)

        grass = regrow_grass(grass, obstacles, tick, params)

        history[-1]["eaten_sheep"] = eaten

    df_history = pd.DataFrame(history)

    return df_history, frames, grass, obstacles, sheep, wolves

## 16. Rodando a simulação principal

In [ ]:
df_history, frames, final_grass, final_obstacles, final_sheep, final_wolves = run_simulation(
    params=params_base,
    save_frames=True,
    frame_interval=5,
    export_images=False
)

df_history.head()

## 17. Estado inicial e final

In [ ]:
first_frame = frames[0]

plot_world(
    first_frame["grass"],
    first_frame["obstacles"],
    first_frame["sheep"],
    first_frame["wolves"],
    tick=first_frame["tick"],
    params=params_base,
    title="Estado inicial"
)

plot_world(
    final_grass,
    final_obstacles,
    final_sheep,
    final_wolves,
    tick=int(df_history["tick"].iloc[-1]),
    params=params_base,
    title="Estado final"
)

## 18. Animação temporal

Esta animação usa marcadores clássicos, não emojis, para funcionar em ambientes sem fonte emoji.

In [ ]:
def animate_simulation(frames, params=params_base):
    fig, ax = plt.subplots(figsize=(8, 8))

    def update(frame):
        ax.clear()

        grass = frame["grass"]
        obstacles = frame["obstacles"]
        sheep = frame["sheep"]
        wolves = frame["wolves"]
        tick = frame["tick"]

        ax.imshow(
            grass.T,
            origin="lower",
            cmap="Greens",
            vmin=0,
            vmax=params["grass_max"],
            alpha=0.95
        )

        ox, oy = np.where(obstacles)
        if len(ox) > 0:
            ax.scatter(ox, oy, s=16, c="black", marker="s", zorder=2)

        healthy_sheep = [s for s in sheep if not s.sick]
        sick_sheep = [s for s in sheep if s.sick]
        healthy_wolves = [w for w in wolves if not w.sick]
        sick_wolves = [w for w in wolves if w.sick]

        if healthy_sheep:
            ax.scatter(
                [s.x for s in healthy_sheep],
                [s.y for s in healthy_sheep],
                s=28,
                c="white",
                edgecolors="black",
                linewidths=0.5,
                marker="o",
                label=f"Ovelhas ({len(sheep)})",
                zorder=3
            )

        if sick_sheep:
            ax.scatter(
                [s.x for s in sick_sheep],
                [s.y for s in sick_sheep],
                s=32,
                c="yellow",
                edgecolors="black",
                linewidths=0.5,
                marker="o",
                zorder=4
            )

        if healthy_wolves:
            ax.scatter(
                [w.x for w in healthy_wolves],
                [w.y for w in healthy_wolves],
                s=70,
                c="red",
                edgecolors="black",
                linewidths=0.6,
                marker="*",
                label=f"Lobos ({len(wolves)})",
                zorder=5
            )

        if sick_wolves:
            ax.scatter(
                [w.x for w in sick_wolves],
                [w.y for w in sick_wolves],
                s=75,
                c="purple",
                edgecolors="black",
                linewidths=0.6,
                marker="*",
                zorder=6
            )

        ax.set_title(f"Wolf Sheep ampliado — tick {tick} — {get_season(tick, params)}")
        ax.set_xlabel("Coordenada X — posição horizontal")
        ax.set_ylabel("Coordenada Y — posição vertical")
        ax.set_xlim(-0.5, grass.shape[0] - 0.5)
        ax.set_ylim(-0.5, grass.shape[1] - 0.5)
        ax.legend(loc="upper right", fontsize=8, framealpha=0.9)

    anim = FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=250,
        repeat=False
    )

    plt.close(fig)
    return anim


anim = animate_simulation(frames, params=params_base)
HTML(anim.to_jshtml())

## 19. Gráficos principais

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df_history["tick"], df_history["sheep"], label="Ovelhas")
plt.plot(df_history["tick"], df_history["wolves"], label="Lobos")
plt.title("Dinâmica populacional")
plt.xlabel("Tick")
plt.ylabel("População")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df_history["tick"], df_history["mean_grass"], label="Grama média")
plt.plot(df_history["tick"], df_history["grass_growth_rate"], label="Taxa de crescimento da grama")
plt.title("Grama e estações")
plt.xlabel("Tick")
plt.ylabel("Valor")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df_history["tick"], df_history["mean_sheep_energy"], label="Energia média das ovelhas")
plt.plot(df_history["tick"], df_history["mean_wolf_energy"], label="Energia média dos lobos")
plt.title("Energia média")
plt.xlabel("Tick")
plt.ylabel("Energia")
plt.legend()
plt.grid(True)
plt.show()

## 20. Métricas de doença, fome e idade

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df_history["tick"], df_history["disease_prevalence_sheep"], label="Prevalência em ovelhas")
plt.plot(df_history["tick"], df_history["disease_prevalence_wolves"], label="Prevalência em lobos")
plt.title("Prevalência de doença")
plt.xlabel("Tick")
plt.ylabel("Proporção doente")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df_history["tick"], df_history["mean_sheep_hunger"], label="Fome média das ovelhas")
plt.plot(df_history["tick"], df_history["mean_wolf_hunger"], label="Fome média dos lobos")
plt.title("Fome média")
plt.xlabel("Tick")
plt.ylabel("Fome média")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df_history["tick"], df_history["mean_sheep_age"], label="Idade média das ovelhas")
plt.plot(df_history["tick"], df_history["mean_wolf_age"], label="Idade média dos lobos")
plt.title("Idade média")
plt.xlabel("Tick")
plt.ylabel("Idade média")
plt.legend()
plt.grid(True)
plt.show()

## 21. Comparação com Lotka-Volterra

O modelo Lotka-Volterra é um sistema de equações diferenciais clássico para dinâmica predador-presa.

Ele é mais abstrato do que o modelo baseado em agentes, pois não considera:

- Espaço.
- Obstáculos.
- Energia individual.
- Idade.
- Doença.
- Migração.
- Estações.
- Estratégias comportamentais.

Mesmo assim, serve como referência matemática.

In [ ]:
def simulate_lotka_volterra(
    prey0,
    predator0,
    alpha=0.08,
    beta=0.002,
    delta=0.0015,
    gamma=0.06,
    dt=1.0,
    steps=300
):
    prey = [prey0]
    predator = [predator0]

    for _ in range(steps):
        x = prey[-1]
        y = predator[-1]

        dx = alpha * x - beta * x * y
        dy = delta * x * y - gamma * y

        prey.append(max(0, x + dx * dt))
        predator.append(max(0, y + dy * dt))

    return pd.DataFrame({
        "tick": np.arange(steps + 1),
        "lv_prey": prey,
        "lv_predator": predator
    })


df_lv = simulate_lotka_volterra(
    prey0=df_history["sheep"].iloc[0],
    predator0=df_history["wolves"].iloc[0],
    steps=int(df_history["tick"].max())
)

df_compare = df_history.merge(df_lv, on="tick", how="left")

plt.figure(figsize=(12, 5))
plt.plot(df_compare["tick"], df_compare["sheep"], label="Ovelhas — modelo baseado em agentes")
plt.plot(df_compare["tick"], df_compare["wolves"], label="Lobos — modelo baseado em agentes")
plt.plot(df_compare["tick"], df_compare["lv_prey"], linestyle="--", label="Presas — Lotka-Volterra")
plt.plot(df_compare["tick"], df_compare["lv_predator"], linestyle="--", label="Predadores — Lotka-Volterra")
plt.title("Comparação: modelo baseado em agentes × Lotka-Volterra")
plt.xlabel("Tick")
plt.ylabel("População")
plt.legend()
plt.grid(True)
plt.show()

## 22. Exportação para CSV

In [ ]:
output_dir = Path("wolf_sheep_outputs")
output_dir.mkdir(exist_ok=True)

history_csv = output_dir / "wolf_sheep_ampliado_historico.csv"
compare_csv = output_dir / "wolf_sheep_comparacao_lotka_volterra.csv"

df_history.to_csv(history_csv, index=False, encoding="utf-8-sig")
df_compare.to_csv(compare_csv, index=False, encoding="utf-8-sig")

history_csv, compare_csv

## 23. Exportação de imagens por tick

Para exportar imagens da simulação, rode a célula abaixo.

Atenção: isso pode gerar muitos arquivos PNG.

In [ ]:
# Exemplo com poucos frames para teste.
# Altere export_images=True na simulação completa se quiser salvar mais imagens.

df_img, frames_img, grass_img, obstacles_img, sheep_img, wolves_img = run_simulation(
    params={**params_base, "n_steps": 40, "random_seed": 123},
    save_frames=True,
    frame_interval=10,
    export_images=True,
    image_dir="wolf_sheep_outputs/frames_png"
)

print("Imagens exportadas em:", "wolf_sheep_outputs/frames_png")

## 24. Experimentos tipo NetLogo BehaviorSpace

Aqui rodamos vários cenários automaticamente, variando:

- Densidade de obstáculos.
- Visão dos lobos.
- Crescimento de grama.
- Probabilidade de doença inicial.

In [ ]:
def run_behaviorspace():
    results = []
    experiment_id = 0

    obstacle_values = [0.02, 0.06, 0.12]
    wolf_vision_values = [2, 5, 8]
    grass_growth_values = [0.25, 0.45, 0.80]
    disease_values = [0.00, 0.02, 0.06]

    for obstacle_density in obstacle_values:
        for wolf_vision in wolf_vision_values:
            for grass_growth in grass_growth_values:
                for disease_prob in disease_values:
                    experiment_id += 1

                    params = params_base.copy()
                    params.update({
                        "obstacle_density": obstacle_density,
                        "wolf_vision": wolf_vision,
                        "grass_regrowth_base": grass_growth,
                        "initial_disease_prob": disease_prob,
                        "n_steps": 180,
                        "random_seed": 10000 + experiment_id,
                    })

                    df_exp, _, _, _, _, _ = run_simulation(
                        params=params,
                        save_frames=False,
                        export_images=False
                    )

                    results.append({
                        "experiment_id": experiment_id,
                        "obstacle_density": obstacle_density,
                        "wolf_vision": wolf_vision,
                        "grass_regrowth_base": grass_growth,
                        "initial_disease_prob": disease_prob,
                        "ticks": int(df_exp["tick"].max()),
                        "final_sheep": int(df_exp["sheep"].iloc[-1]),
                        "final_wolves": int(df_exp["wolves"].iloc[-1]),
                        "max_sheep": int(df_exp["sheep"].max()),
                        "max_wolves": int(df_exp["wolves"].max()),
                        "mean_sheep": float(df_exp["sheep"].mean()),
                        "mean_wolves": float(df_exp["wolves"].mean()),
                        "mean_grass": float(df_exp["mean_grass"].mean()),
                        "mean_disease_sheep": float(df_exp["disease_prevalence_sheep"].mean()),
                        "mean_disease_wolves": float(df_exp["disease_prevalence_wolves"].mean()),
                        "total_eaten_sheep": int(df_exp["eaten_sheep"].sum()),
                        "sheep_extinct_final": bool(df_exp["sheep"].iloc[-1] == 0),
                        "wolves_extinct_final": bool(df_exp["wolves"].iloc[-1] == 0),
                    })

    return pd.DataFrame(results)


df_behaviorspace = run_behaviorspace()
df_behaviorspace.head()

In [ ]:
behaviorspace_csv = output_dir / "wolf_sheep_behaviorspace_ampliado.csv"
df_behaviorspace.to_csv(behaviorspace_csv, index=False, encoding="utf-8-sig")

df_behaviorspace.groupby("wolf_vision")[["final_sheep", "final_wolves", "total_eaten_sheep"]].mean()

In [ ]:
plt.figure(figsize=(12, 5))
for vision in sorted(df_behaviorspace["wolf_vision"].unique()):
    subset = df_behaviorspace[df_behaviorspace["wolf_vision"] == vision]
    grouped = subset.groupby("obstacle_density")["final_sheep"].mean()
    plt.plot(grouped.index, grouped.values, marker="o", label=f"Visão dos lobos = {vision}")

plt.title("BehaviorSpace: efeito dos obstáculos e visão dos lobos nas ovelhas finais")
plt.xlabel("Densidade de obstáculos")
plt.ylabel("Média de ovelhas finais")
plt.legend()
plt.grid(True)
plt.show()

## 25. Protótipo de dashboard com Streamlit

A célula abaixo cria um arquivo `streamlit_wolf_sheep_dashboard.py`.

Para executar no terminal:

```bash
streamlit run streamlit_wolf_sheep_dashboard.py
```

O dashboard lê o CSV exportado e mostra gráficos interativos.

In [ ]:
streamlit_code = """
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

st.set_page_config(page_title="Wolf Sheep Simple 3 Ampliado", layout="wide")

st.title("Wolf Sheep Simple 3 Ampliado")
st.write("Dashboard para análise dos resultados exportados pelo notebook.")

csv_path = Path("wolf_sheep_outputs/wolf_sheep_ampliado_historico.csv")

if not csv_path.exists():
    st.error("Arquivo CSV não encontrado. Rode primeiro o notebook para gerar os resultados.")
    st.stop()

df = pd.read_csv(csv_path)

st.subheader("Dados")
st.dataframe(df.head())

col1, col2 = st.columns(2)

with col1:
    st.metric("Ticks simulados", int(df["tick"].max()))
    st.metric("Ovelhas finais", int(df["sheep"].iloc[-1]))
    st.metric("Lobos finais", int(df["wolves"].iloc[-1]))

with col2:
    st.metric("Máximo de ovelhas", int(df["sheep"].max()))
    st.metric("Máximo de lobos", int(df["wolves"].max()))
    st.metric("Total de ovelhas predadas", int(df["eaten_sheep"].sum()))

st.subheader("Populações")
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["tick"], df["sheep"], label="Ovelhas")
ax.plot(df["tick"], df["wolves"], label="Lobos")
ax.set_xlabel("Tick")
ax.set_ylabel("População")
ax.grid(True)
ax.legend()
st.pyplot(fig)

st.subheader("Grama e estações")
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["tick"], df["mean_grass"], label="Grama média")
ax.plot(df["tick"], df["grass_growth_rate"], label="Taxa de crescimento")
ax.set_xlabel("Tick")
ax.grid(True)
ax.legend()
st.pyplot(fig)

st.subheader("Doença")
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["tick"], df["disease_prevalence_sheep"], label="Ovelhas doentes")
ax.plot(df["tick"], df["disease_prevalence_wolves"], label="Lobos doentes")
ax.set_xlabel("Tick")
ax.set_ylabel("Prevalência")
ax.grid(True)
ax.legend()
st.pyplot(fig)

st.subheader("Tabela completa")
st.dataframe(df)
"""

streamlit_path = Path("streamlit_wolf_sheep_dashboard.py")
streamlit_path.write_text(streamlit_code, encoding="utf-8")
streamlit_path

## 26. Síntese interpretativa

A ampliação do modelo mostra como uma dinâmica predador-presa simples pode se tornar muito mais rica quando adicionamos:

- Estrutura espacial.
- Comportamento adaptativo.
- Limites fisiológicos.
- Doença.
- Migração.
- Estações.
- Obstáculos.

Esses elementos tornam o sistema mais próximo de uma situação ecológica real e evidenciam propriedades típicas de Sistemas Complexos:

- Emergência.
- Não linearidade.
- Feedbacks positivos e negativos.
- Dependência das condições iniciais.
- Sensibilidade a parâmetros.
- Colapso e recuperação.
- Auto-organização espacial.